# Notebook 5 — Exploratory generalization beyond income

This notebook asks whether the **Anna Karenina pattern in life-satisfaction dispersion generalizes beyond household income**.

The main paper's confirmatory analysis remains the income analysis in Notebook 4. Everything here is **exploratory / appendix analysis**. The purpose is to test whether structured heterogeneity also appears across other respondent characteristics and whether the six LLMs reproduce those patterns despite general variance compression.

## Characteristics examined

1. **Employment status (Q279)**  
   - Full category-specific profiles are reported.
   - A targeted contrast compares **unemployed respondents (7)** with **currently employed respondents (1–3: full-time, part-time, self-employed)**.
   - Retired people, homemakers, students, and "other" are retained in the category-level analysis but excluded from the targeted employed-versus-unemployed contrast.

2. **Education (Q275)**  
   - Full ordered profile from 0 (early childhood/no education) to 8 (doctoral/equivalent).
   - A summary contrast compares **lower-secondary-or-less (0–2)** with **tertiary education (5–8)**.
   - Categories 3–4 remain in the full profile but are not used in that summary contrast.

3. **Marital status (Q273)**  
   - Full category-specific profiles are reported.
   - A descriptive summary compares **not partnered (3–6)** with **married/living together (1–2)**.
   - Because marital status is not an ordered socioeconomic scale, this comparison is treated as exploratory rather than as a directional Anna Karenina test.

4. **Perceived freedom and control (Q48)**  
   - Full ordered profile from 1 (no choice at all) to 10 (a great deal of choice).
   - A summary contrast compares **low control (1–3)** with **high control (8–10)**.

## Scale-free normalization

For characteristic \(v\), category \(j\), and source \(m\),

\[
D_{jmv}=\frac{SD_{jmv}}{\sum_j w_{jv}SD_{jmv}},
\]

where \(w_{jv}\) is the **human sample share** in category \(j\) of characteristic \(v\). The same human weights are used for humans and every LLM.

As in the main income analysis, this removes each source's overall dispersion scale for that characteristic and asks whether the **relative location of heterogeneity** is preserved.

## Outputs

The notebook saves:
- variable and category audits;
- raw means, SDs, variances, and normalized SDs;
- targeted scale-free contrasts;
- country-fixed-effect RIF-variance regressions;
- categorical country-FE RIF estimates;
- appendix-ready vector PDF figures.

No API calls are made. Existing frozen predictions are reused unchanged.


In [ ]:
# Run once in Deepnote if needed.
%pip install -q pandas numpy scipy statsmodels matplotlib


In [ ]:
from pathlib import Path
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

DATA_DIR = Path("output/full_wvs")
PROFILE_FILE = DATA_DIR / "profiles_for_prediction.csv"
PRED_DIR = DATA_DIR / "predictions"
LOCKED_DIR = DATA_DIR / "analysis_locked"
FROZEN_FILE = LOCKED_DIR / "00_frozen_analysis_dataset.csv"

OUTDIR = DATA_DIR / "analysis_generalization"
OUTDIR.mkdir(parents=True, exist_ok=True)

EXPECTED_N = 93901
EXPECTED_COUNTRIES = 66

MODEL_FILES = {
    "GPT-5.6 Luna": PRED_DIR / "openai_gpt56_luna_FULL.csv",
    "Claude Sonnet 5": PRED_DIR / "anthropic_sonnet5_FULL.csv",
    "Gemini 3.8 Flash": PRED_DIR / "google_gemini38_flash_FULL.csv",
    "DeepSeek V4.1 Flash": PRED_DIR / "deepseek_v41_flash_FULL.csv",
    "Gemma 4 31B": PRED_DIR / "deepinfra_gemma4_31b_FULL.csv",
    "Qwen 3.7 Plus": PRED_DIR / "alibaba_qwen37_plus_FULL.csv",
}

SLUG = {
    "Human": "human",
    "GPT-5.6 Luna": "gpt56_luna",
    "Claude Sonnet 5": "claude_sonnet5",
    "Gemini 3.8 Flash": "gemini38_flash",
    "DeepSeek V4.1 Flash": "deepseek_v41_flash",
    "Gemma 4 31B": "gemma4_31b",
    "Qwen 3.7 Plus": "qwen37_plus",
}
SOURCE_ORDER = ["Human"] + list(MODEL_FILES)

VARIABLES = ["Q279", "Q275", "Q273", "Q48"]

LABELS = {
    "Q279": {
        1: "Full-time employee", 2: "Part-time employee", 3: "Self-employed",
        4: "Retired/pensioned", 5: "Homemaker", 6: "Student",
        7: "Unemployed", 8: "Other"
    },
    "Q275": {
        0: "Early childhood/no education", 1: "Primary", 2: "Lower secondary",
        3: "Upper secondary", 4: "Post-secondary non-tertiary",
        5: "Short-cycle tertiary", 6: "Bachelor", 7: "Master", 8: "Doctoral"
    },
    "Q273": {
        1: "Married", 2: "Living together", 3: "Divorced",
        4: "Separated", 5: "Widowed", 6: "Single"
    },
    "Q48": {i: str(i) for i in range(1, 11)},
}

VAR_TITLES = {
    "Q279": "Employment status",
    "Q275": "Educational attainment",
    "Q273": "Marital status",
    "Q48": "Perceived freedom and control",
}

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("Output directory:", OUTDIR)


## 1. Load the frozen six-model analysis sample and recover the additional respondent characteristics

The preferred input is Notebook 4's frozen merged dataset, which guarantees that Notebook 5 uses exactly the same 93,901 respondents and exactly the same six LLM predictions.

Q48 and Q279 were supplied to the LLMs but were not retained in Notebook 4's reduced frozen analysis file. They are merged back from `profiles_for_prediction.csv` using `WVS_ROW_ID`. Q273 and Q275 are also checked against the profile file.


In [ ]:
if not PROFILE_FILE.exists():
    raise FileNotFoundError(
        f"{PROFILE_FILE} not found. Notebook 5 expects the full profiles file produced by Notebook 1."
    )

profiles = pd.read_csv(
    PROFILE_FILE,
    usecols=lambda c: c in {
        "WVS_ROW_ID", "B_COUNTRY_ALPHA", "COUNTRY_NAME",
        "Q49", "Q288", "Q48", "Q273", "Q275", "Q279"
    },
    low_memory=False
)

profiles["WVS_ROW_ID"] = pd.to_numeric(profiles["WVS_ROW_ID"], errors="raise").astype(int)
for c in ["Q49", "Q288", "Q48", "Q273", "Q275", "Q279"]:
    if c in profiles:
        profiles[c] = pd.to_numeric(profiles[c], errors="coerce")
        profiles.loc[profiles[c] < 0, c] = np.nan

# Prefer the exact frozen Notebook 4 dataset.
if FROZEN_FILE.exists():
    analysis = pd.read_csv(FROZEN_FILE, low_memory=False)
    analysis["WVS_ROW_ID"] = pd.to_numeric(analysis["WVS_ROW_ID"], errors="raise").astype(int)

    # Merge only the additional variables needed here. For Q273/Q275, compare
    # against the frozen values before keeping the frozen version.
    extra = profiles[["WVS_ROW_ID", "Q48", "Q279", "Q273", "Q275"]].copy()
    chk = analysis.merge(extra, on="WVS_ROW_ID", how="left", validate="one_to_one",
                         suffixes=("", "_profile"))

    for c in ["Q273", "Q275"]:
        cp = f"{c}_profile"
        if c in chk and cp in chk:
            same = (
                chk[c].fillna(-999999).astype(float)
                == chk[cp].fillna(-999999).astype(float)
            )
            if not same.all():
                raise AssertionError(f"{c} differs between frozen analysis and profile file.")

    analysis = chk.drop(columns=[c for c in ["Q273_profile", "Q275_profile"] if c in chk])

else:
    print("Frozen Notebook 4 dataset not found; reconstructing it from profiles and prediction files.")
    req = ["WVS_ROW_ID", "Q49", "Q288", "B_COUNTRY_ALPHA", "COUNTRY_NAME"]
    miss = [c for c in req if c not in profiles]
    if miss:
        raise ValueError(f"Missing profile columns: {miss}")

    analysis = profiles.copy()
    analysis["country"] = analysis["B_COUNTRY_ALPHA"].astype(str)
    analysis["country_name"] = analysis["COUNTRY_NAME"].astype(str)

    ids = set(analysis["WVS_ROW_ID"])
    for model, path in MODEL_FILES.items():
        if not path.exists():
            raise FileNotFoundError(f"{model}: {path}")
        p = pd.read_csv(path, low_memory=False)
        if not {"WVS_ROW_ID", "prediction"}.issubset(p.columns):
            raise ValueError(f"{model}: missing WVS_ROW_ID/prediction")
        p["WVS_ROW_ID"] = pd.to_numeric(p["WVS_ROW_ID"], errors="raise").astype(int)
        p["prediction"] = pd.to_numeric(p["prediction"], errors="raise")
        if set(p["WVS_ROW_ID"]) != ids:
            raise AssertionError(f"{model}: respondent IDs do not match profiles.")
        col = "ls_" + SLUG[model]
        analysis = analysis.merge(
            p[["WVS_ROW_ID", "prediction"]].rename(columns={"prediction": col}),
            on="WVS_ROW_ID", how="left", validate="one_to_one"
        )

# Harmonize country columns.
if "country" not in analysis:
    analysis["country"] = profiles.set_index("WVS_ROW_ID").loc[
        analysis["WVS_ROW_ID"], "B_COUNTRY_ALPHA"
    ].astype(str).values
if "country_name" not in analysis:
    analysis["country_name"] = profiles.set_index("WVS_ROW_ID").loc[
        analysis["WVS_ROW_ID"], "COUNTRY_NAME"
    ].astype(str).values

# Technical audit.
pred_cols = {m: "ls_" + SLUG[m] for m in MODEL_FILES}
source_to_col = {"Human": "Q49", **pred_cols}

assert len(analysis) == EXPECTED_N, (len(analysis), EXPECTED_N)
assert analysis["WVS_ROW_ID"].is_unique
assert analysis["Q49"].between(1, 10).all()
assert analysis["Q288"].between(1, 10).all()
assert analysis["country"].nunique() == EXPECTED_COUNTRIES

for s, col in source_to_col.items():
    if col not in analysis:
        raise ValueError(f"Missing outcome column for {s}: {col}")
    if not pd.to_numeric(analysis[col], errors="coerce").between(1, 10).all():
        raise AssertionError(f"{s}: invalid life-satisfaction predictions.")

audit = pd.DataFrame({
    "item": ["N", "countries", "unique_WVS_ROW_ID"],
    "value": [len(analysis), analysis["country"].nunique(), analysis["WVS_ROW_ID"].nunique()]
})
audit.to_csv(OUTDIR / "00_generalization_input_audit.csv", index=False)

print("TECHNICAL AUDIT PASSED")
display(audit)


## 2. Variable audit and category definitions

This section reports valid sample sizes before any substantive analysis. No categories are chosen based on the observed LLM results.


In [ ]:
expected_ranges = {
    "Q279": set(range(1, 9)),
    "Q275": set(range(0, 9)),
    "Q273": set(range(1, 7)),
    "Q48": set(range(1, 11)),
}

audit_rows = []
category_rows = []

for var in VARIABLES:
    x = pd.to_numeric(analysis[var], errors="coerce")
    valid = x.dropna()
    observed = set(valid.astype(int).unique())
    if not observed.issubset(expected_ranges[var]):
        raise ValueError(f"{var}: unexpected codes {sorted(observed - expected_ranges[var])}")

    audit_rows.append({
        "variable": var,
        "description": VAR_TITLES[var],
        "n_valid": int(valid.size),
        "pct_valid": 100 * valid.size / len(analysis),
        "n_categories_observed": len(observed),
        "codes_observed": ",".join(map(str, sorted(observed))),
    })

    vc = valid.astype(int).value_counts().sort_index()
    for code, n in vc.items():
        category_rows.append({
            "variable": var,
            "description": VAR_TITLES[var],
            "category": int(code),
            "category_label": LABELS[var].get(int(code), str(code)),
            "n": int(n),
            "pct_of_valid": 100 * n / valid.size,
        })

var_audit = pd.DataFrame(audit_rows)
cat_audit = pd.DataFrame(category_rows)

var_audit.to_csv(OUTDIR / "01a_generalization_variable_audit.csv", index=False)
cat_audit.to_csv(OUTDIR / "01b_generalization_category_audit.csv", index=False)

display(var_audit.round(2))
display(cat_audit.round(2))


## 3. Category-specific means and dispersion

For each characteristic, category, and source, calculate:
- sample size;
- mean life satisfaction;
- raw SD;
- variance;
- normalized SD.

The normalization uses **fixed human category shares** for each characteristic, exactly paralleling the logic of the primary income normalization.


In [ ]:
moment_rows = []
normalized_rows = []
scale_rows = []

for var in VARIABLES:
    dvar = analysis.loc[analysis[var].notna()].copy()
    dvar[var] = dvar[var].astype(int)

    # Fixed human category shares for this variable.
    cats = sorted(dvar[var].unique())
    counts = dvar[var].value_counts().reindex(cats).astype(float)
    weights = counts / counts.sum()

    for source in SOURCE_ORDER:
        col = source_to_col[source]

        gm = (
            dvar.groupby(var)[col]
            .agg(n="size", mean_ls="mean", sd_ls="std", variance_ls="var")
            .reindex(cats)
        )

        scale = float(np.sum(weights.values * gm["sd_ls"].values))
        scale_rows.append({
            "variable": var,
            "source": source,
            "weighted_mean_category_sd": scale
        })

        for cat in cats:
            row = gm.loc[cat]
            rec = {
                "variable": var,
                "variable_label": VAR_TITLES[var],
                "category": int(cat),
                "category_label": LABELS[var].get(int(cat), str(cat)),
                "source": source,
                "n": int(row["n"]),
                "mean_ls": float(row["mean_ls"]),
                "sd_ls": float(row["sd_ls"]),
                "variance_ls": float(row["variance_ls"]),
                "human_category_weight": float(weights.loc[cat]),
                "scale": scale,
                "normalized_sd": float(row["sd_ls"] / scale),
            }
            moment_rows.append(rec)
            normalized_rows.append({
                k: rec[k] for k in [
                    "variable", "variable_label", "category", "category_label",
                    "source", "n", "human_category_weight", "sd_ls", "scale", "normalized_sd"
                ]
            })

moments = pd.DataFrame(moment_rows)
normalized = pd.DataFrame(normalized_rows)
scales = pd.DataFrame(scale_rows)

moments.to_csv(OUTDIR / "02_generalization_category_moments.csv", index=False)
normalized.to_csv(OUTDIR / "03_generalization_normalized_sd_profiles.csv", index=False)
scales.to_csv(OUTDIR / "03a_generalization_normalization_scales.csv", index=False)

display(moments.head(30).round(4))


## 4. Targeted scale-free contrasts

These contrasts provide a compact cross-domain summary.

- **Employment:** unemployed (Q279=7) versus employed (Q279=1–3).
- **Education:** lower-secondary-or-less (Q275=0–2) versus tertiary (Q275=5–8).
- **Marital status:** not partnered (Q273=3–6) versus partnered (Q273=1–2). Descriptive only.
- **Perceived control:** low control (Q48=1–3) versus high control (Q48=8–10).

For each source, the main dispersion statistic is \(SD_A/SD_B\). A value above 1 means greater dispersion in comparison group A. Because the numerator and denominator come from the same source, a source-wide multiplicative compression factor cancels from this ratio.


In [ ]:
CONTRASTS = {
    "Employment: unemployed vs employed": {
        "variable": "Q279",
        "A_label": "Unemployed",
        "B_label": "Employed",
        "A_codes": [7],
        "B_codes": [1, 2, 3],
        "directional_AK": True,
    },
    "Education: lower-secondary-or-less vs tertiary": {
        "variable": "Q275",
        "A_label": "Lower secondary or less",
        "B_label": "Tertiary",
        "A_codes": [0, 1, 2],
        "B_codes": [5, 6, 7, 8],
        "directional_AK": True,
    },
    "Marital status: not partnered vs partnered": {
        "variable": "Q273",
        "A_label": "Not partnered",
        "B_label": "Partnered",
        "A_codes": [3, 4, 5, 6],
        "B_codes": [1, 2],
        "directional_AK": False,
    },
    "Perceived control: low vs high": {
        "variable": "Q48",
        "A_label": "Low control",
        "B_label": "High control",
        "A_codes": [1, 2, 3],
        "B_codes": [8, 9, 10],
        "directional_AK": True,
    },
}

contrast_rows = []

for contrast_name, spec in CONTRASTS.items():
    var = spec["variable"]
    for source in SOURCE_ORDER:
        col = source_to_col[source]
        A = analysis.loc[analysis[var].isin(spec["A_codes"]), col].astype(float)
        B = analysis.loc[analysis[var].isin(spec["B_codes"]), col].astype(float)

        sdA, sdB = A.std(ddof=1), B.std(ddof=1)
        contrast_rows.append({
            "contrast": contrast_name,
            "variable": var,
            "source": source,
            "A": spec["A_label"],
            "B": spec["B_label"],
            "n_A": len(A),
            "n_B": len(B),
            "mean_A": A.mean(),
            "mean_B": B.mean(),
            "mean_difference_A_minus_B": A.mean() - B.mean(),
            "sd_A": sdA,
            "sd_B": sdB,
            "sd_ratio_A_to_B": sdA / sdB,
            "log_sd_ratio": np.log(sdA / sdB),
            "directional_AK_comparison": spec["directional_AK"],
        })

contrasts = pd.DataFrame(contrast_rows)
contrasts.to_csv(OUTDIR / "04_generalization_targeted_contrasts.csv", index=False)

display(
    contrasts[
        ["contrast", "source", "n_A", "n_B", "mean_difference_A_minus_B", "sd_ratio_A_to_B"]
    ].round(4)
)


## 5. Country-fixed-effect RIF-variance robustness

This section asks whether the cross-characteristic patterns remain after absorbing country fixed effects.

The RIF for variance is implemented as in Notebook 4:

\[
RIF_i^{Var}=(LS_i-\bar{LS})^2.
\]

Models use country-clustered standard errors.

Specifications:
- employment: unemployed indicator among employed/unemployed respondents only;
- education: Q275 entered linearly as an ordered summary;
- marital status: not-partnered indicator versus partnered;
- perceived control: Q48 entered linearly.

For education and perceived control, the linear coefficient is a summary of the ordered pattern; the full categorical estimates are saved separately below.


In [ ]:
rif_specs = {
    "Employment": {
        "variable": "Q279",
        "type": "binary",
        "keep_codes": [1, 2, 3, 7],
        "make_x": lambda s: s.eq(7).astype(int),
        "x_name": "Unemployed (vs employed)",
    },
    "Education": {
        "variable": "Q275",
        "type": "linear",
        "keep_codes": list(range(0, 9)),
        "make_x": lambda s: s.astype(float),
        "x_name": "Education category (0–8)",
    },
    "Marital status": {
        "variable": "Q273",
        "type": "binary",
        "keep_codes": list(range(1, 7)),
        "make_x": lambda s: s.isin([3, 4, 5, 6]).astype(int),
        "x_name": "Not partnered (vs partnered)",
    },
    "Perceived control": {
        "variable": "Q48",
        "type": "linear",
        "keep_codes": list(range(1, 11)),
        "make_x": lambda s: s.astype(float),
        "x_name": "Perceived control (1–10)",
    },
}

rif_rows = []

for spec_name, spec in rif_specs.items():
    var = spec["variable"]

    for source in SOURCE_ORDER:
        col = source_to_col[source]
        d = analysis[["country", var, col]].copy()
        d = d[d[var].isin(spec["keep_codes"])].dropna().rename(columns={col: "ls"})
        d["x"] = spec["make_x"](d[var])
        d["rifv"] = (d["ls"] - d["ls"].mean()) ** 2

        m = smf.ols("rifv ~ x + C(country)", data=d).fit(
            cov_type="cluster", cov_kwds={"groups": d["country"]}
        )

        rif_rows.append({
            "specification": spec_name,
            "variable": var,
            "source": source,
            "x_definition": spec["x_name"],
            "n": len(d),
            "countries": d["country"].nunique(),
            "coef": m.params["x"],
            "se": m.bse["x"],
            "p": m.pvalues["x"],
        })

rif_summary = pd.DataFrame(rif_rows)
rif_summary.to_csv(OUTDIR / "05_generalization_RIF_country_FE_summary.csv", index=False)

display(rif_summary.round(5))


### Full categorical country-FE RIF estimates

These estimates avoid imposing linearity on education or perceived control and retain all employment and marital-status categories. The first observed category is the reference group for each characteristic.


In [ ]:
rif_cat_rows = []

for var in VARIABLES:
    valid_codes = sorted(
        pd.to_numeric(analysis[var], errors="coerce").dropna().astype(int).unique()
    )
    ref = valid_codes[0]

    for source in SOURCE_ORDER:
        col = source_to_col[source]
        d = analysis[["country", var, col]].dropna().rename(columns={col: "ls"}).copy()
        d[var] = d[var].astype(int)
        d["rifv"] = (d["ls"] - d["ls"].mean()) ** 2

        m = smf.ols(f"rifv ~ C({var}) + C(country)", data=d).fit(
            cov_type="cluster", cov_kwds={"groups": d["country"]}
        )

        for code in valid_codes:
            term = f"C({var})[T.{code}]"
            rif_cat_rows.append({
                "variable": var,
                "variable_label": VAR_TITLES[var],
                "source": source,
                "category": code,
                "category_label": LABELS[var].get(code, str(code)),
                "reference_category": ref,
                "coef_vs_reference": 0.0 if code == ref else m.params.get(term, np.nan),
                "se": 0.0 if code == ref else m.bse.get(term, np.nan),
                "p": np.nan if code == ref else m.pvalues.get(term, np.nan),
                "n_model": len(d),
                "countries": d["country"].nunique(),
            })

rif_categorical = pd.DataFrame(rif_cat_rows)
rif_categorical.to_csv(OUTDIR / "06_generalization_RIF_country_FE_categorical.csv", index=False)

display(rif_categorical.head(40).round(5))


## 6. Appendix figures

All figures are saved as **vector PDF only**.

Ordered characteristics (education and perceived control) are shown as connected profiles. Unordered characteristics (employment and marital status) are shown as category-specific points without connecting lines so that the figure does not imply an ordinal scale.

The Human series is visually emphasized.


In [ ]:
def save_ordered_profile(var, filename, xlabel):
    d = normalized[normalized["variable"].eq(var)].copy()

    # Extra vertical room for the education panel so the x-axis title
    # and legend do not collide.
    figsize = (8.8, 6.0) if var == "Q275" else (8.2, 5.5)
    fig, ax = plt.subplots(figsize=figsize)

    for source in SOURCE_ORDER:
        g = d[d["source"].eq(source)].sort_values("category")
        if source == "Human":
            ax.plot(g["category"], g["normalized_sd"], marker="o",
                    linewidth=2.8, markersize=5.2, label=source)
        else:
            ax.plot(g["category"], g["normalized_sd"], marker="o",
                    linewidth=1.6, markersize=3.8, label=source)

    ax.axhline(1.0, linestyle="--", linewidth=1.0, alpha=0.45)
    ax.set_title(
        f"Normalized life-satisfaction dispersion by {VAR_TITLES[var].lower()}",
        loc="center", fontweight="bold", pad=12
    )
    ax.set_xlabel(xlabel, labelpad=14 if var == "Q275" else 8)
    ax.set_ylabel("Normalized SD of life satisfaction", labelpad=10)
    ax.set_xticks(sorted(d["category"].unique()))
    ax.tick_params(axis="x", pad=6)
    ax.grid(axis="y", alpha=0.20)

    if var == "Q275":
        ax.legend(
            loc="upper center", bbox_to_anchor=(0.5, -0.24),
            ncol=4, frameon=False, fontsize=8.5
        )
        fig.subplots_adjust(left=0.13, right=0.98, top=0.90, bottom=0.32)
    else:
        ax.legend(
            loc="upper center", bbox_to_anchor=(0.5, -0.16),
            ncol=4, frameon=False, fontsize=8.5
        )
        fig.subplots_adjust(bottom=0.27)

    fig.savefig(OUTDIR / filename, format="pdf", bbox_inches="tight")
    plt.show()


def save_unordered_profile(var, filename, xlabel):
    d = normalized[normalized["variable"].eq(var)].copy()
    cats = sorted(d["category"].unique())
    labels = [LABELS[var].get(int(c), str(c)) for c in cats]

    # Employment has long category labels, so give the labels, x-axis title,
    # and legend separate vertical space.
    figsize = (10.2, 6.4) if var == "Q279" else (9.4, 5.8)
    fig, ax = plt.subplots(figsize=figsize)
    offsets = np.linspace(-0.24, 0.24, len(SOURCE_ORDER))

    for offset, source in zip(offsets, SOURCE_ORDER):
        g = d[d["source"].eq(source)].set_index("category").reindex(cats)
        x = np.arange(len(cats)) + offset
        if source == "Human":
            ax.scatter(x, g["normalized_sd"], s=55, label=source, zorder=4)
        else:
            ax.scatter(x, g["normalized_sd"], s=28, label=source, alpha=0.9)

    ax.axhline(1.0, linestyle="--", linewidth=1.0, alpha=0.45)
    ax.set_title(
        f"Normalized life-satisfaction dispersion by {VAR_TITLES[var].lower()}",
        loc="center", fontweight="bold", pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel("Normalized SD of life satisfaction", labelpad=10)
    ax.set_xticks(np.arange(len(cats)))
    ax.set_xticklabels(
        labels,
        rotation=30 if var == "Q279" else 28,
        ha="right",
        rotation_mode="anchor"
    )
    ax.tick_params(axis="x", pad=7)
    ax.grid(axis="y", alpha=0.20)

    if var == "Q279":
        ax.legend(
            loc="upper center", bbox_to_anchor=(0.5, -0.35),
            ncol=4, frameon=False, fontsize=8.5
        )
        fig.subplots_adjust(left=0.13, right=0.98, top=0.90, bottom=0.47)
    else:
        ax.legend(
            loc="upper center", bbox_to_anchor=(0.5, -0.25),
            ncol=4, frameon=False, fontsize=8.5
        )
        fig.subplots_adjust(bottom=0.36)

    fig.savefig(OUTDIR / filename, format="pdf", bbox_inches="tight")
    plt.show()


save_unordered_profile(
    "Q279", "figA1_employment_normalized_sd.pdf", "Employment status"
)
save_ordered_profile(
    "Q275", "figA2_education_normalized_sd.pdf",
    "Highest educational level (0 = lowest, 8 = highest)"
)
save_unordered_profile(
    "Q273", "figA3_marital_status_normalized_sd.pdf", "Marital status"
)
save_ordered_profile(
    "Q48", "figA4_perceived_control_normalized_sd.pdf",
    "Perceived freedom and control (1 = lowest, 10 = highest)"
)

print("Saved vector PDFs:")
for p in sorted(OUTDIR.glob("figA*.pdf")):
    print(" -", p.name)


## 7. Compact appendix-ready summary

This final table is intended as a convenient diagnostic rather than a new confirmatory test. It reports each source's scale-free SD ratio for the four targeted contrasts and the corresponding mean-life-satisfaction difference.


In [ ]:
summary_wide = contrasts.pivot(
    index="source",
    columns="contrast",
    values="sd_ratio_A_to_B"
).reindex(SOURCE_ORDER)

summary_wide.to_csv(OUTDIR / "07_generalization_SD_ratio_summary_wide.csv")

mean_wide = contrasts.pivot(
    index="source",
    columns="contrast",
    values="mean_difference_A_minus_B"
).reindex(SOURCE_ORDER)

mean_wide.to_csv(OUTDIR / "08_generalization_mean_difference_summary_wide.csv")

print("SD ratio A/B (values > 1 indicate greater dispersion in group A):")
display(summary_wide.round(3))

print("Mean LS difference A - B:")
display(mean_wide.round(3))


## 8. Interpretation guardrails

When reporting these results:

- Treat this notebook as **exploratory appendix evidence**, not as an extension of the preregistered primary test.
- Do not interpret a raw SD difference as structural fidelity without considering general LLM under-dispersion. Use normalized profiles or within-source SD ratios for cross-source interpretation.
- Employment and marital-status categories are not ordinal. Do not fit or interpret a linear category trend for them.
- The employed-versus-unemployed contrast excludes retirees, homemakers, students, and "other"; these groups remain visible in the full employment profile.
- The education and perceived-control linear RIF coefficients are compact summaries. The categorical RIF output should be consulted when the relationship is visibly nonlinear.
- The marital partnered/not-partnered comparison is descriptive. It should not be described as a causal effect of marriage or partnership.
- All associations are observational. Greater dispersion among disadvantaged groups does not identify heterogeneous causal treatment effects.
- If the patterns generalize, the defensible conclusion is that preservation of **relative heterogeneity structure** is not unique to income. Exact profile equality should not be claimed without a dedicated equality test.


In [ ]:
# ============================================================
# PACKAGE NOTEBOOK 5 RESULTS FOR DOWNLOAD
# ============================================================

from pathlib import Path
import zipfile

RESULTS_DIR = Path("output/full_wvs/analysis_generalization")
ZIP_PATH = Path("Notebook5_generalization_results.zip")

# Remove old copy if it exists
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# Create ZIP in the project root so it appears in Deepnote Files
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(RESULTS_DIR.rglob("*")):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(RESULTS_DIR))

# Verify the ZIP
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad_file = zf.testzip()
    files = zf.namelist()

assert bad_file is None, f"ZIP verification failed: {bad_file}"

print(f"✓ Created: {ZIP_PATH}")
print(f"✓ Files included: {len(files)}")
print(f"✓ Size: {ZIP_PATH.stat().st_size / 1024**2:.2f} MB")

print("\\nContents:")
for f in files:
    print("  ", f)

print("\\nDownload Notebook5_generalization_results.zip from the Deepnote Files sidebar.")

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=7aaa7215-b731-433d-9b62-8be4a70a4410' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>